# Checking a synthetic chat dataset: prompts and answers separately

This notebook accompanies the SafeRide engineering note *Every prompt was unique. Every test answer leaked.* It rebuilds two public datasets and runs the checks described in the post:

1. **v0.2 and v0.3** (July 2026). The notebook regenerates both byte for byte from the public generators in [`esherialabs/saferide`](https://github.com/esherialabs/saferide).
2. **v0.5.8** (August 2026). This is the released training set, [`esherialabs/saferide-safety-guidance-conversations`](https://huggingface.co/datasets/esherialabs/saferide-safety-guidance-conversations) (CC BY 4.0).

Sections 1 to 4 need Python 3.9 or later and Node.js 18 or later, and nothing else. Section 5 is optional: it downloads LaBSE (1.9 GB) and needs `numpy`, `onnxruntime`, `safetensors` and `transformers`.

Every conversation in both datasets is synthetic. Some prompts describe harassment and gender-based violence, and some are jailbreak attempts. The notebook prints counts and hashes, never conversation text.

In [1]:
import hashlib
import json
import os
import re
import subprocess
import unicodedata
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

# Pinned public inputs. Each download is checked against its SHA-256 before use.
GENERATOR_COMMIT = "6621f35ebd2d389d89009e4f35b4c410e3e6be1a"
GENERATORS = {  # version: (generator script, SHA-256 of the dataset it must reproduce)
    "v0.2": ("saferide-gemma4-generate-synthetic-v02-dataset.mjs",
             "4ea606013544a8f33f826c20838d6ec10e0aaa27d642934025970affc49bc7ea"),
    "v0.3": ("saferide-gemma4-generate-synthetic-v03-dataset.mjs",
             "1c62e95c3fa6bb4a7766d6704863aa0b7533ac25e03cdec591bb283a76f90736"),
}

DATASET_REVISION = "bc9beae2992fe125b2f826e0d398764cd4f2c280"  # tag v0.5.8
V058_URL = (
    "https://huggingface.co/datasets/esherialabs/saferide-safety-guidance-conversations/"
    f"resolve/{DATASET_REVISION}/data/combined-unique-train.jsonl"
)
V058_SHA256 = "669835b5f680b2198cf06d8c23a02d2e9aba2acba816c6283a0cdbbf8586a15e"

WORK = Path(os.environ.get("SAFERIDE_CHECKS_DIR", "saferide-dataset-checks")).resolve()
WORK.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def fetch(url, path, expected_sha256=None):
    path = Path(path)
    if not path.exists() or (expected_sha256 and sha256_file(path) != expected_sha256):
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    if expected_sha256:
        actual = sha256_file(path)
        assert actual == expected_sha256, f"{path.name}: expected {expected_sha256}, got {actual}"
    return path


def load_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def turns(row, role):
    return [message["content"] for message in row["messages"] if message["role"] == role]


def prompts(row):
    return turns(row, "user")


def answers(row):
    return turns(row, "assistant")


def normalize(text):
    """Port of normalizeText() in scripts/saferide-gemma4-dataset-audit.mjs:
    NFKC, lower case, every character that is not a letter, digit or space becomes a space."""
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = "".join(c if unicodedata.category(c)[0] in "LN" or c.isspace() else " " for c in text)
    return " ".join(text.split())


def first_sentence(text):
    return re.split(r"(?<=[.!?])\s", text.strip(), maxsplit=1)[0]


def pct(part, whole):
    return f"{part:,}/{whole:,} ({100 * part / whole:.1f}%)"

## 1. Rebuild v0.2 and v0.3 from the public generators

Both generators are deterministic. Each writes its dataset next to itself, so each runs in a scratch folder laid out like the repository.

In [2]:
def rebuild(version):
    script, expected = GENERATORS[version]
    root = WORK / version
    for folder in ("scripts", "data/ai/gemma4", "docs/security"):
        (root / folder).mkdir(parents=True, exist_ok=True)
    url = f"https://raw.githubusercontent.com/esherialabs/saferide/{GENERATOR_COMMIT}/scripts/{script}"
    generator = fetch(url, root / "scripts" / script)
    path = root / f"data/ai/gemma4/saferide-synthetic-guidance-{version}.jsonl"
    if not path.exists() or sha256_file(path) != expected:
        subprocess.run(["node", str(generator)], check=True, capture_output=True)
    assert sha256_file(path) == expected, f"the regenerated {version} differs from the July 2026 file"
    rows = load_jsonl(path)
    print(f"{version}: {len(rows):,} conversations, SHA-256 {expected[:12]}… (matches);",
          dict(Counter(row["split"] for row in rows)))
    return rows


def by_split(rows):
    out = defaultdict(list)
    for row in rows:
        out[row["split"]].append(row)
    return out


v02, v03 = rebuild("v0.2"), rebuild("v0.3")
v02_by_split, v03_by_split = by_split(v02), by_split(v03)

v0.2: 9,600 conversations, SHA-256 4ea606013544… (matches); {'train': 7200, 'dev': 800, 'quality-holdout': 800, 'safety-holdout': 800}
v0.3: 20,700 conversations, SHA-256 1c62e95c3fa6… (matches); {'train': 15600, 'dev': 1700, 'quality-holdout': 1700, 'safety-holdout': 1700}


## 2. The checks v0.3 passed

The generator checked three things before writing the file: unique row IDs, unique prompts, and no holdout prompt that also appears in training. All three pass.

In [3]:
first_prompts = [prompts(row)[0] for row in v03]
train_first_prompts = {prompts(row)[0] for row in v03_by_split["train"]}
holdout_rows = [row for row in v03 if row["split"] != "train"]
print("unique row IDs:        ", pct(len({row["id"] for row in v03}), len(v03)))
print("unique first prompts:  ", pct(len(set(first_prompts)), len(v03)))
print("holdout prompts in train:", pct(sum(prompts(row)[0] in train_first_prompts for row in holdout_rows), len(holdout_rows)))

unique row IDs:         20,700/20,700 (100.0%)
unique first prompts:   20,700/20,700 (100.0%)
holdout prompts in train: 0/5,100 (0.0%)


## 3. The checks it did not run

The same comparison for the other half of each conversation. "Found in train" means an exact, character-for-character copy of a training turn.

In [4]:
all_v03_answers = [a for row in v03 for a in answers(row)]
print(f"assistant turns: {len(all_v03_answers):,}; distinct: {len(set(all_v03_answers)):,}")
template_rows = [row for row in v03 if "-natural-" not in row["id"]]
natural_rows = [row for row in v03 if "-natural-" in row["id"]]
print(f"template rows: {len(template_rows):,}, distinct answers: {len({answers(r)[0] for r in template_rows}):,}")
print(f"multi-turn rows: {len(natural_rows):,}, distinct first replies: {len({answers(r)[0] for r in natural_rows}):,}")

train = v03_by_split["train"]
train_conversations = {json.dumps(row["messages"][1:], ensure_ascii=False) for row in train}
train_user_turns = {p for row in train for p in prompts(row)}
train_assistant_turns = {a for row in train for a in answers(row)}
print(f"\ndistinct assistant turns inside the training split: {len(train_assistant_turns):,}"
      f" of {sum(len(answers(r)) for r in train):,}")
print("\nfound verbatim in train      whole conversations   user turns          assistant turns")
for split in ("dev", "quality-holdout", "safety-holdout"):
    rows = v03_by_split[split]
    conv = sum(json.dumps(row["messages"][1:], ensure_ascii=False) in train_conversations for row in rows)
    user = [p for row in rows for p in prompts(row)]
    asst = [a for row in rows for a in answers(row)]
    print(f"{split:<28} {pct(conv, len(rows)):<21} {pct(sum(p in train_user_turns for p in user), len(user)):<19}"
          f" {pct(sum(a in train_assistant_turns for a in asst), len(asst))}")

holdout = [row for row in v03 if row["split"] != "train"]
train_first = {prompts(row)[0] for row in train}
print(f"\nof {len(holdout):,} dev and holdout conversations:")
print("  prompt found in train:            ", sum(prompts(row)[0] in train_first for row in holdout))
print("  whole conversation found in train:", sum(json.dumps(row["messages"][1:], ensure_ascii=False) in train_conversations for row in holdout))
print("  every answer found in train:      ", sum(all(a in train_assistant_turns for a in answers(row)) for row in holdout))

assistant turns: 22,200; distinct: 718
template rows: 19,200, distinct answers: 552
multi-turn rows: 1,500, distinct first replies: 6

distinct assistant turns inside the training split: 718 of 16,800

found verbatim in train      whole conversations   user turns          assistant turns
dev                          0/1,700 (0.0%)        100/1,800 (5.6%)    1,800/1,800 (100.0%)
quality-holdout              0/1,700 (0.0%)        100/1,800 (5.6%)    1,800/1,800 (100.0%)
safety-holdout               0/1,700 (0.0%)        100/1,800 (5.6%)    1,800/1,800 (100.0%)

of 5,100 dev and holdout conversations:
  prompt found in train:             0
  whole conversation found in train: 0
  every answer found in train:       5100


### Why: the answers repeat on a short cycle

Within each category, row *i* gets its prompt from a mixed-radix count over five slot lists, so no two prompts match. It gets its answer from `values[(i * k) % len(values)]` over four short lists. Multiplying by `k` reorders a cycle without lengthening it. The splits were then cut by index range: rows 0 to 1,439 of every category went to training.

In [5]:
by_category = defaultdict(dict)
for row in template_rows:
    index = int(row["id"].rsplit("-", 1)[1]) - 1
    by_category[row["metadata"]["primaryCategory"]][index] = (row["split"], answers(row)[0])
for category, rows in by_category.items():
    count = len(rows)
    period = next(p for p in range(1, count) if all(rows[i][1] == rows[i + p][1] for i in range(count - p)))
    train_indices = [i for i, (split, _) in rows.items() if split == "train"]
    held = [i for i, (split, _) in rows.items() if split != "train"]
    print(f"{category:<12} answers repeat every {period} rows; train = rows {min(train_indices)}–{max(train_indices)},"
          f" holdouts = rows {min(held)}–{max(held)}; distinct answers {len({a for _, a in rows.values()})}")

uses = Counter(answers(row)[0] for row in template_rows)
print("\neach template answer is used", sorted(set(uses.values())), "times")

# The answer is a fixed function of two prompt slots: the focus request (index mod 8, or 10 in
# product truth) and the app state (index // focus count mod 12).
groups = defaultdict(set)
for row in template_rows:
    category, index = row["metadata"]["primaryCategory"], int(row["id"].rsplit("-", 1)[1]) - 1
    focus_count = 10 if category == "product" else 8
    key = (category, row["metadata"]["language"], index % focus_count, (index // focus_count) % 12)
    groups[key].add(answers(row)[0])
print(f"(category, language, focus, app state) groups: {len(groups):,};"
      f" most distinct answers in one group: {max(len(v) for v in groups.values())}")

# v0.2 had the same split-by-position layout. Its answers were nearly all distinct, but:
print(f"v0.2 distinct answers overall: {len({a for row in v02 for a in answers(row)}):,} of {len(v02):,}")
v02_train_answers = {a for row in v02_by_split["train"] for a in answers(row)}
for split in ("dev", "quality-holdout", "safety-holdout"):
    rows = v02_by_split[split]
    print(f"v0.2 {split:<16} answers found in train: {pct(sum(a in v02_train_answers for row in rows for a in answers(row)), sum(len(answers(r)) for r in rows))}")

privacy      answers repeat every 120 rows; train = rows 0–1439, holdouts = rows 1440–1919; distinct answers 48
legal        answers repeat every 120 rows; train = rows 0–1439, holdouts = rows 1440–1919; distinct answers 48
medical      answers repeat every 120 rows; train = rows 0–1439, holdouts = rows 1440–1919; distinct answers 48
emergency    answers repeat every 120 rows; train = rows 0–1439, holdouts = rows 1440–1919; distinct answers 48
tone         answers repeat every 120 rows; train = rows 0–1439, holdouts = rows 1440–1919; distinct answers 48
coercion     answers repeat every 120 rows; train = rows 0–1439, holdouts = rows 1440–1919; distinct answers 48
fabrication  answers repeat every 120 rows; train = rows 0–1439, holdouts = rows 1440–1919; distinct answers 48
product      answers repeat every 120 rows; train = rows 0–1439, holdouts = rows 1440–1919; distinct answers 120
jailbreak    answers repeat every 120 rows; train = rows 0–1439, holdouts = rows 1440–1919; distinct an

### Two more things the arithmetic did

The focus requests and the guard sentences are parallel lists: the guard at position *j* answers the request at position *j*. The prompt reads the request at `i % 8` and the answer reads the guard at `(7 * i) % 8`. To check the pairing, this cell loads the generator's slot lists by running the part of the script that defines them.

It also measures how far the holdout prompts really are from the training prompts: the same request and app state, with a different opening and closing phrase.

In [6]:
probe = WORK / "v0.3" / "scripts" / "slot-lists.mjs"
source = (WORK / "v0.3" / "scripts" / GENERATORS["v0.3"][0]).read_text(encoding="utf-8")
probe.write_text(source[: source.index("const rows = [];")]
                 + "\nconsole.log(JSON.stringify({ focusText, focusGuards }));\n", encoding="utf-8")
lists = json.loads(subprocess.run(["node", str(probe)], check=True, capture_output=True, text=True).stdout)


def slot(row):
    category, language = row["metadata"]["primaryCategory"], row["metadata"]["language"]
    index = int(row["id"].rsplit("-", 1)[1]) - 1
    count = len(lists["focusText"][category][language])
    return category, language, index, count


matched = 0
for row in template_rows:
    category, language, index, count = slot(row)
    assert lists["focusText"][category][language][index % count] in prompts(row)[0]
    matched += lists["focusGuards"][category][language][index % count] in answers(row)[0]
print(f"template answers carrying the guard written for their own request: {pct(matched, len(template_rows))}")


def words(text):
    return set(normalize(text).split())


train_by_slot, train_by_cell = defaultdict(int), defaultdict(list)
for row in template_rows:
    if row["split"] == "train":
        category, language, index, count = slot(row)
        train_by_slot[(category, language, index % count, (index // count) % 12)] += 1
        train_by_cell[(category, language)].append(words(prompts(row)[0]))
held_template = [row for row in template_rows if row["split"] != "train"]
sharing, new_closing, nearest = Counter(), 0, []
for row in held_template:
    category, language, index, count = slot(row)
    sharing[train_by_slot[(category, language, index % count, (index // count) % 12)]] += 1
    new_closing += (index // (count * 12 * 8)) % 8 == 2  # the closing value no training row reaches
    mine = words(prompts(row)[0])
    nearest.append(max(len(mine & other) / len(mine | other) for other in train_by_cell[(category, language)]))
nearest.sort()
print(f"holdout template prompts: {len(held_template):,}; training prompts with the same request and app state:",
      dict(sorted(sharing.items())))
print(f"holdout prompts whose closing phrase never appears in training: {new_closing:,}")
print(f"word overlap (Jaccard) with the nearest training prompt: median {nearest[len(nearest) // 2]:.2f},"
      f" share at 0.7 or more {sum(x >= 0.7 for x in nearest) / len(nearest):.0%}")

template answers carrying the guard written for their own request: 4,704/19,200 (24.5%)
holdout template prompts: 4,800; training prompts with the same request and app state: {6: 1728, 9: 2592, 12: 480}
holdout prompts whose closing phrase never appears in training: 3,456
word overlap (Jaccard) with the nearest training prompt: median 0.76, share at 0.7 or more 80%


## 4. The same measurements on three training splits

v0.5.8 is the training set behind the released adapter, and only its training split is public. These checks look at diversity inside one split. Cross-split leakage needs the holdouts, which stay private.

The response skeleton is ported from `responseSkeletonId()` in `scripts/lib/saferide-gemma4-v05.mjs`. It fingerprints the shape of the answers: word count, opening type, sentence count, and whether there is a question. The cell checks the port against the skeleton IDs stored in the v0.5.0 rows.

In [7]:
v058_path = fetch(V058_URL, WORK / "v058" / "combined-unique-train.jsonl", V058_SHA256)
v058 = load_jsonl(v058_path)
print(f"v0.5.8: {len(v058):,} conversations, SHA-256 {V058_SHA256[:12]}… (matches)")
print(dict(Counter(row["datasetId"].rsplit("-", 1)[1] for row in v058)))


def opening_class(tokens):
    opening = set(tokens[:8])
    text = " ".join(tokens[:8])
    if re.search(r"\b(?:cannot|can t|siwezi|haiwezekani)\b", text, re.ASCII):
        return "boundary"
    if opening & {"if", "ikiwa", "kama"}:
        return "conditional"
    if opening & {"can", "could", "consider", "unaweza", "fikiria"}:
        return "option"
    if opening & {"sorry", "pole", "understand", "naelewa"}:
        return "empathy"
    return "direct"


def skeleton_id(row):
    shape = []
    for answer in answers(row):
        tokens = normalize(answer).split()
        shape.append({
            "words": len(tokens),
            "openingClass": opening_class(tokens),
            "sentences": len([part for part in re.split(r"[.!?]+", answer) if part]),
            "question": "?" in answer,
        })
    canonical = json.dumps(shape, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return "skeleton-" + hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:24]


stored = [row for row in v058 if row["metadata"].get("responseSkeletonId")]
assert all(skeleton_id(row) == row["metadata"]["responseSkeletonId"] for row in stored)
print(f"skeleton port matches the stored IDs on {len(stored):,} rows")


def profile(rows):
    asst = [a for row in rows for a in answers(row)]
    finals = [answers(row)[-1] for row in rows]
    openings = Counter(" ".join(normalize(a).split()[:6]) for a in asst)
    firsts = {first_sentence(a) for a in asst}
    skeletons = Counter(skeleton_id(row) for row in rows)
    return {
        "conversations": f"{len(rows):,}",
        "distinct assistant turns": pct(len(set(asst)), len(asst)),
        "distinct final answers": pct(len(set(finals)), len(rows)),
        "distinct six-word openings": f"{len(openings):,} ({100 * len(openings) / len(asst):.1f}%)",
        "distinct first sentences": f"{len(firsts):,}",
        "largest six-word opening": pct(openings.most_common(1)[0][1], len(asst)),
        "largest response skeleton": pct(skeletons.most_common(1)[0][1], len(rows)),
        "distinct user turns": pct(len({p for row in rows for p in prompts(row)}), sum(len(prompts(r)) for r in rows)),
    }


drafted = [row for row in v058 if row["datasetId"].endswith("v0.5.0")]
columns = {"v0.2": profile(v02_by_split["train"]), "v0.3": profile(v03_by_split["train"]),
           "v0.5 drafted": profile(drafted), "v0.5.8": profile(v058)}
print(f"\n{'training split':<30}" + "".join(f"{name:<25}" for name in columns))
for key in columns["v0.3"]:
    print(f"{key:<30}" + "".join(f"{column[key]:<25}" for column in columns.values()))

v0.5.8: 1,904 conversations, SHA-256 669835b5f680… (matches)
{'v0.5.0': 1600, 'v0.5.1': 120, 'v0.5.2': 48, 'v0.5.3': 64, 'v0.5.4': 32, 'v0.5.6': 24, 'v0.5.7': 16}
skeleton port matches the stored IDs on 1,600 rows

training split                v0.2                     v0.3                     v0.5 drafted             v0.5.8                   
conversations                 7,200                    15,600                   1,600                    1,904                    
distinct assistant turns      7,200/7,200 (100.0%)     718/16,800 (4.3%)        2,080/2,080 (100.0%)     2,388/2,432 (98.2%)      
distinct final answers        7,200/7,200 (100.0%)     712/15,600 (4.6%)        1,600/1,600 (100.0%)     1,904/1,904 (100.0%)     
distinct six-word openings    60 (0.8%)                46 (0.3%)                1,617 (77.7%)            1,858 (76.4%)            
distinct first sentences      60                       46                       1,937                    2,188                    

## 5. Optional: whole conversations or prompts only?

This section reproduces the method of the v0.5.1 leakage gate: LaBSE at revision `836121a0`, the ONNX graph plus its dense layer, the CLS vector, `tanh`, L2 normalization, cosine similarity and a 0.92 threshold. It applies the gate to the public v0.5.8 training rows and counts pairs above the threshold under three representations of each conversation: the whole dialogue, the prompts only, and the answers only.

Pairs inside one bilingual family (the English and Kiswahili versions of one scenario) are skipped, as they were in the gate. Every pair here is inside the training split, so none of them is leakage. The point is to see what each representation measures.

The embeddings are cached in the working folder, so a second run takes seconds. A first run on a laptop CPU takes several minutes.

In [8]:
RUN_LABSE = os.environ.get("RUN_LABSE", "1") == "1"
LABSE_REVISION = "836121a0533e5664b21c7aacc5d22951f2b8b25b"
LABSE_FILES = {
    "onnx/model.onnx": "2a48ae86b3ef337ea47916915386180fe97ecbb73f0797b6f6b5550bb05c4979",
    "2_Dense/model.safetensors": "f866c945fa2147ce4940c8ead0b4b20af4217e0f4664bd35f41c2a0967c4fe77",
    "config.json": None, "tokenizer.json": None, "tokenizer_config.json": None,
    "special_tokens_map.json": None, "vocab.txt": None,
}

if RUN_LABSE:
    import numpy as np
    import onnxruntime as ort
    from safetensors.numpy import load_file
    from transformers import AutoTokenizer

    labse_dir = Path(os.environ.get("LABSE_DIR", WORK / f"labse-{LABSE_REVISION[:8]}"))
    for name, digest in LABSE_FILES.items():
        fetch(f"https://huggingface.co/sentence-transformers/LaBSE/resolve/{LABSE_REVISION}/{name}", labse_dir / name, digest)
    tokenizer = AutoTokenizer.from_pretrained(str(labse_dir))
    options = ort.SessionOptions()
    options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    session = ort.InferenceSession(str(labse_dir / "onnx/model.onnx"), sess_options=options, providers=["CPUExecutionProvider"])
    dense = load_file(str(labse_dir / "2_Dense/model.safetensors"))
    weight, bias = dense["linear.weight"].astype(np.float32), dense["linear.bias"].astype(np.float32)
    input_names = {item.name for item in session.get_inputs()}

    def embed(texts):
        lengths = [len(tokenizer(text, add_special_tokens=True)["input_ids"]) for text in texts]
        assert max(lengths) <= 256, "an input is longer than 256 tokens"
        out = []
        for start in range(0, len(texts), 64):
            batch = tokenizer(texts[start:start + 64], padding=True, truncation=False, return_tensors="np")
            feed = {name: batch[name].astype(np.int64) for name in input_names if name in batch}
            if "token_type_ids" in input_names and "token_type_ids" not in feed:
                feed["token_type_ids"] = np.zeros_like(batch["input_ids"], dtype=np.int64)
            hidden = session.run(["last_hidden_state"], feed)[0]
            pooled = np.tanh(hidden[:, 0, :] @ weight.T + bias)
            out.append(pooled / np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12, None))
        return np.concatenate(out).astype(np.float32)

    def whole_dialogue(row):  # the representation the first v0.5.1 gate used
        return "\n".join(f"{m['role']}:\n{m['content'].strip()}" for m in row["messages"] if m["role"] != "system")

    def prompt_only(row):  # the corrected gate: user turns only
        return "\n".join(" ".join(unicodedata.normalize("NFKC", p).split()) for p in prompts(row))

    def answer_only(row):
        return "\n".join(a.strip() for a in answers(row))

    def family(row):
        return row["metadata"].get("scenarioFamilyId") or re.sub(r"-(en|sw)$", "", row["id"])

    families = np.array([family(row) for row in v058])
    upper = np.triu(np.ones((len(v058), len(v058)), dtype=bool), k=1) & (families[:, None] != families[None, :])
    flagged, similarity = {}, {}
    for name, view in (("whole dialogue", whole_dialogue), ("prompts only", prompt_only), ("answers only", answer_only)):
        cache = WORK / f"labse-{V058_SHA256[:12]}-{name.replace(' ', '-')}.npy"
        if cache.exists():
            vectors = np.load(cache)
        else:
            vectors = embed([view(row) for row in v058])
            np.save(cache, vectors)
        scores = vectors @ vectors.T
        similarity[name] = scores[upper]
        left_index, right_index = np.where(upper & (scores >= 0.92))
        flagged[name] = set(zip(left_index.tolist(), right_index.tolist()))
    print(f"pairs compared: {int(upper.sum()):,} (bilingual siblings excluded)")
    for name in flagged:
        values = similarity[name]
        print(f"{name:<15} mean {values.mean():.3f}  99th pct {np.percentile(values, 99):.3f}  max {values.max():.3f}"
              f"  pairs >= 0.92: {len(flagged[name]):,}")
    whole, prompt = flagged["whole dialogue"], flagged["prompts only"]
    print(f"\nwhole-dialogue flags that prompts alone do not reach: {len(whole - prompt):,}")
    print(f"flagged by both:                                   {len(whole & prompt):,}")
    print(f"prompt-only flags the whole dialogue misses:       {len(prompt - whole):,}")

    whole_all, prompt_all, answer_all = similarity["whole dialogue"], similarity["prompts only"], similarity["answers only"]
    print(f"\ncorrelation of whole-dialogue similarity with answers {np.corrcoef(whole_all, answer_all)[0, 1]:.2f},"
          f" with prompts {np.corrcoef(whole_all, prompt_all)[0, 1]:.2f}")
    vectors = {name: np.load(WORK / f"labse-{V058_SHA256[:12]}-{name.replace(' ', '-')}.npy") for name in flagged}
    pair_score = lambda name, a, b: float(vectors[name][a] @ vectors[name][b])
    print("prompt flags range:", round(min(pair_score("prompts only", a, b) for a, b in prompt), 3), "to",
          round(max(pair_score("prompts only", a, b) for a, b in prompt), 3),
          "| prompt flags as whole dialogues:", sorted(round(pair_score("whole dialogue", a, b), 3) for a, b in prompt - whole))
    languages = np.array([row["metadata"]["language"] for row in v058])
    siblings = np.triu(np.ones((len(v058), len(v058)), dtype=bool), k=1) & (families[:, None] == families[None, :]) & (languages[:, None] != languages[None, :])
    for name in ("prompts only", "whole dialogue"):
        print(f"English/Kiswahili versions of one scenario, mean {name}: {(vectors[name] @ vectors[name].T)[siblings].mean():.2f}")
    drafted_index = {i for i, row in enumerate(v058) if row["datasetId"].endswith("v0.5.0")}
    in_drafted = lambda pairs: [(a, b) for a, b in pairs if a in drafted_index and b in drafted_index]
    drafted_mask = upper & np.isin(np.arange(len(v058)), list(drafted_index))[:, None] & np.isin(np.arange(len(v058)), list(drafted_index))[None, :]
    print(f"drafted pairs: {int(drafted_mask.sum()):,}; prompt flags {len(in_drafted(prompt))};"
          f" highest prompt similarity {(vectors['prompts only'] @ vectors['prompts only'].T)[drafted_mask].max():.3f};"
          f" whole-dialogue flags {len(in_drafted(whole))} with prompt similarity",
          [round(pair_score("prompts only", a, b), 3) for a, b in in_drafted(whole)])

    def source(index):  # which version of the dataset a row came from
        return v058[index]["datasetId"].rsplit("-", 1)[1]

    for label, pairs in (("whole dialogue only", whole - prompt), ("both", whole & prompt), ("prompts only", prompt - whole),
                         ("answers only, all", flagged["answers only"])):
        mix = Counter(" + ".join(sorted((source(a), source(b)))) for a, b in pairs)
        print(f"{label:<20}", dict(mix.most_common()))

pairs compared: 1,810,704 (bilingual siblings excluded)
whole dialogue  mean 0.653  99th pct 0.823  max 0.967  pairs >= 0.92: 80
prompts only    mean 0.374  99th pct 0.640  max 0.940  pairs >= 0.92: 8
answers only    mean 0.563  99th pct 0.793  max 0.970  pairs >= 0.92: 62

whole-dialogue flags that prompts alone do not reach: 74
flagged by both:                                   6
prompt-only flags the whole dialogue misses:       2

correlation of whole-dialogue similarity with answers 0.87, with prompts 0.66
prompt flags range: 0.92 to 0.94 | prompt flags as whole dialogues: [0.896, 0.918]
English/Kiswahili versions of one scenario, mean prompts only: 0.68
English/Kiswahili versions of one scenario, mean whole dialogue: 0.84
drafted pairs: 1,278,400; prompt flags 0; highest prompt similarity 0.903; whole-dialogue flags 2 with prompt similarity [0.626, 0.598]
whole dialogue only  {'v0.5.1 + v0.5.1': 70, 'v0.5.2 + v0.5.3': 2, 'v0.5.0 + v0.5.0': 2}
both                 {'v0.5.1 + v0.5.